In [19]:
import json
from pathlib import Path

INPUT_PATH = Path("../../infra/json/kg_extraction/bellicum_contract_kg.json")
OUTPUT_DIR = Path("../../infra/json/kg_extraction")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

In [20]:
import re
from collections import defaultdict

def normalize_text(text):
    text = text.lower().strip()

    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)

    return text

def canonical_id(entity_type, label):
    clean = normalize_text(label)
    clean = clean.replace(" ", "_")

    return f"{entity_type.lower()}::{clean}"

In [21]:
def normalize_entities(entities):
    canonical_map = {}
    new_entities = {}

    for ent in entities:
        label = ent.get("label", "")
        ent_type = ent.get("type", "")

        cid = canonical_id(ent_type, label)

        canonical_map[ent["id"]] = cid

        if cid not in new_entities:
            new_entities[cid] = {
                "id": cid,
                "type": ent_type,
                "label": label,
                "properties": ent.get("properties", {}),
                "evidence_text": [ent.get("evidence_text", "")]
            }
        else:
            new_entities[cid]["evidence_text"].append(
                ent.get("evidence_text", "")
            )

    return list(new_entities.values()), canonical_map

In [22]:
def normalize_relations(relations, canonical_map):
    new_relations = []

    for rel in relations:
        src = rel["source"]
        tgt = rel["target"]

        if src not in canonical_map or tgt not in canonical_map:
            continue

        new_relations.append({
            "source": canonical_map[src],
            "target": canonical_map[tgt],
            "type": rel["type"],
            "confidence": rel.get("confidence", 1.0),
            "evidence_text": rel.get("evidence_text", "")
        })

    return new_relations

In [23]:
ALIAS_MAP = {
    "party::bellicum": "party::bellicum_pharmaceuticals_inc",
    "party::miltenyi": "party::miltenyi_biotec_gmbh",
    "party::miltenyi_biotec": "party::miltenyi_biotec_gmbh",
    "party::the_parties": "definedterm::parties",
    "party::party": "definedterm::party",
    "party::regulatory_authorityies": "party::regulatory_authority",
}

def is_bad_entity(ent):
    return ent["id"] in {"value::", "definedterm::"} or ent["label"].strip() in {"[***]", "[...***...]"}

In [24]:
def apply_alias(entity_id):
    return ALIAS_MAP.get(entity_id, entity_id)


def normalize_kg(data):
    entities = data["knowledge_graph"]["entities"]
    relations = data["knowledge_graph"]["relations"]

    print("Original entities:", len(entities))
    print("Original relations:", len(relations))

    norm_entities, canonical_map = normalize_entities(entities)

    canonical_map = {
        old_id: apply_alias(new_id)
        for old_id, new_id in canonical_map.items()
    }

    merged_entities = {}

    for ent in norm_entities:
        aliased_id = apply_alias(ent["id"])

        if is_bad_entity(ent):
            continue

        if aliased_id not in merged_entities:
            ent["id"] = aliased_id
            merged_entities[aliased_id] = ent
        else:
            merged_entities[aliased_id]["evidence_text"].extend(
                ent.get("evidence_text", [])
            )

            merged_entities[aliased_id]["properties"].update(
                ent.get("properties", {})
            )

    norm_entities = list(merged_entities.values())

    norm_relations = normalize_relations(relations, canonical_map)
    valid_entity_ids = {ent["id"] for ent in norm_entities}

    norm_relations = [
        rel for rel in norm_relations
        if rel["source"] in valid_entity_ids and rel["target"] in valid_entity_ids
    ]

    print("Normalized entities:", len(norm_entities))
    print("Normalized relations:", len(norm_relations))

    return {
        "mode": "knowledge_graph_normalized",
        "knowledge_graph": {
            "entities": norm_entities,
            "relations": norm_relations
        }
    }

In [25]:
normalized_kg = normalize_kg(data)

OUTPUT_PATH = OUTPUT_DIR / "bellicum_contract_kg_normalized.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(normalized_kg, f, indent=2, ensure_ascii=False)


Original entities: 1864
Original relations: 2795
Normalized entities: 1499
Normalized relations: 2738
